# Graduate Lab: Fairness as Large-Scale Multiobjective Optimization (Student Version)

This lab studies a practical question in fair machine learning:

> Can we improve a fairness criterion by changing how different demographic groups enter the training objective?

You will work with a three-group ACSIncome classification task and the **BADR** toolbox (`badr`). The notebook provides the data loading, plotting, and optimization drivers. You implement the mathematical pieces that determine what the algorithm is optimizing.

The lower-level problem trains one shared linear classifier with group weights $\lambda\in\Delta^{S-1}$:

$$
w^\star(\lambda)=\arg\min_w f(w,\lambda),
\qquad
f(w,\lambda)=\sum_{s=1}^{S}\lambda_s\,\ell_s(w).
$$

Each choice of $\lambda$ gives a fitted model. The outer objective evaluates a fairness metric on that fitted model:

$$
V(\lambda)=\mathrm{metric}\big(w^\star(\lambda)\big).
$$

BADR-GD searches for group weights $\lambda$ that reduce $V(\lambda)$.

In this notebook you will:

1. implement differentiable fairness metrics;
2. compare basic group-reweighting heuristics;
3. recover the group weights induced by a min-max baseline;
4. implement the derivative oracles used by BADR-GD;
5. interpret the learned weights on a simplex plot.

Complete the code cells that end with `raise NotImplementedError`. The surrounding text gives the formulas, expected shapes, and checks needed to debug each step.

<a id="toc"></a>
## Table of Contents
1. [Setup and Data](#sec-setup)
2. [Question 1: Fairness Metrics](#sec-metrics)
3. [Question 2: Reweighting](#sec-reweight)
4. [Question 3: Min-Max Reweighting](#sec-minmax)
5. [Question 4: BADR Oracle and BADR-GD](#sec-oracle)
6. [Final Experiment and Interpretation](#sec-figure)


> **Reference.** The `badr` documentation covers the dataset loaders, fairness metrics, models, derivative oracles, algorithms, and the high-level `Badr` API:
>
> <https://badr.readthedocs.io/en/latest/>
>
> This lab exposes the lower-level pieces so that the optimization structure is explicit.

## 1. Setup and Data <a class="anchor" id="sec-setup"></a>

The next cell installs BADR and downloads the helper module used by the lab. Run it once at the beginning of a Colab session. If you are working from a local clone with the package and helper file already available, you can skip the installation cell and run the imports directly.

In [ ]:
# Install the BADR toolbox and fetch the lab helpers (Colab-friendly).
!pip install -q badr
!wget -q -O lab_helpers.py https://raw.githubusercontent.com/AdaptiveDecisionMakingGroup/badr/lab/lab_helpers.py

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

import badr            # the toolbox: datasets, models, metrics, oracles, algorithms
import lab_helpers as lh   # given utilities (data, plotting, toolbox glue)

### 1.1. Dataset and Model

We use a binary ACSIncome task from `badr.datasets.fetch_ACSIncome` (the target is whether personal income exceeds 50,000 dollars). We select Alaska in 2018 and merge the sex-by-race buckets into three groups, then standardize the features.

The dataset stores group-specific arrays:

- `X_train_list[s]`: feature matrix for group $s$;
- `y_train_list[s]`: binary labels for group $s$.

The lower-level model is `badr.models.LogisticRegression` **without an intercept**. The parameter vector $w$ is therefore the vector of linear coefficients, and the score of an example $x_i$ is $x_i^\top w$.

For group $s$, the empirical logistic loss is

$$
\ell_s(w)
=
\frac{1}{n_s}\sum_{i\in G_s}
\left[\mathrm{softplus}(x_i^\top w)-y_i\,x_i^\top w\right]
+\frac{\mu}{2}\lVert w\rVert^2.
$$

Training with group weights $\lambda$ solves

$$
w^\star(\lambda)
=
\arg\min_w \sum_{s=1}^{S}\lambda_s\ell_s(w).
$$

In code, `lh.fit_weights(model, dset, lambda)` sets the weights, fits the model, and returns $w^\star(\lambda)$.

Before continuing, check that `n_groups == 3`. The final simplex plots assume three groups.

In [ ]:
dset = badr.datasets.fetch_ACSIncome(states=["AK"], year=2018, n_groups=3)
model = lh.make_model(l2_reg=1e-1)

# per-group training arrays (float64 JAX), used by the metrics you will write
X_list, y_list = lh.group_lists(dset, "train")
n_groups = dset.n_groups

## 2. Question 1: Fairness Metrics <a class="anchor" id="sec-metrics"></a>

Implement three differentiable fairness metrics. Each metric has signature

```python
metric(w, X_list, y_list)
```

and returns a scalar JAX value. Let

$$
s_i=x_i^\top w
$$

be the score of individual $i$.

### 2.1. Individual Fairness

The individual-fairness surrogate penalizes score gaps between people from different groups. Pairs with similar labels get larger weight:

$$
\mathrm{IF}(w)
=
\frac{1}{Z}
\sum_{a<b}
\sum_{i\in G_a}
\sum_{j\in G_b}
\exp(-|y_i-y_j|)\,(s_i-s_j)^2,
\qquad
Z=\sum_{a\ne b} n_a n_b.
$$

For each pair of groups $(a,b)$, form all pairwise score gaps using broadcasting: one vector with shape `(n_a, 1)` and one with shape `(1, n_b)`.

### 2.2. Equalized Odds

Equalized odds compares behavior across groups conditional on the true label. We use soft predictions

$$
\sigma(\rho s_i)
$$

and smooth max/min operators

$$
\widetilde{\max}_\rho(v)=\frac{1}{\rho}\log\sum_k e^{\rho v_k},
\qquad
\widetilde{\min}_\rho(v)=-\widetilde{\max}_\rho(-v).
$$

For each group,

$$
\mathrm{TPR}_s=
\frac{\sum_{i\in G_s}\sigma(\rho s_i)\mathbf 1[y_i=1]}
{\sum_{i\in G_s}\mathbf 1[y_i=1]},
\qquad
\mathrm{FPR}_s=
\frac{\sum_{i\in G_s}\sigma(\rho s_i)\mathbf 1[y_i=0]}
{\sum_{i\in G_s}\mathbf 1[y_i=0]}.
$$

The smooth equalized-odds disparity is

$$
\mathrm{EOdds}(w)
=
\frac12
\left[
\widetilde{\max}_\rho(\mathrm{TPR})-\widetilde{\min}_\rho(\mathrm{TPR})
+
\widetilde{\max}_\rho(\mathrm{FPR})-\widetilde{\min}_\rho(\mathrm{FPR})
\right].
$$

Use $\rho=6$.

### 2.3. Equal Opportunity

Equal opportunity keeps only the true-positive-rate disparity:

$$
\mathrm{EOpp}(w)
=
\widetilde{\max}_\rho(\mathrm{TPR})
-
\widetilde{\min}_\rho(\mathrm{TPR}).
$$

### Implementation

Complete `individual_fairness`, `equalized_odds`, and `equal_opportunity` below.

Useful checks while coding:

- keep arrays in JAX (`jnp`) so the metrics remain differentiable;
- build one length-`S` vector of group rates before applying `soft_max` and `soft_min`;
- if equalized odds fails, print the TPR and FPR vectors first and debug those rates separately.

In [ ]:
def soft_max(v, rho):
    return jax.nn.logsumexp(rho * v) / rho


def soft_min(v, rho):
    return -jax.nn.logsumexp(-rho * v) / rho


def individual_fairness(w, X_list, y_list):
    """Cross-group, similarity-weighted squared score gap (scalar).

    Scores per group:  s = X @ w   (so s_i = x_i^T w).
    Formula:
        IF(w) = (1 / Z) * sum_{a < b} sum_{i in G_a} sum_{j in G_b}
                    exp(-|y_i - y_j|) * (s_i - s_j) ** 2,
        Z = sum_{a != b} n_a * n_b.
    """
    # Steps:
    #   1. s_a = X_list[a] @ w  for every group a.
    #   2. for each pair a < b, broadcast over i, j ([:, None] vs [None, :]):
    #        sim_ij = exp(-|y_a[i] - y_b[j]|),  gap_ij = (s_a[i] - s_b[j]) ** 2;
    #        accumulate sum(sim * gap).
    #   3. divide the running total by Z = sum_{a != b} n_a * n_b.
    raise NotImplementedError


def equalized_odds(w, X_list, y_list, rho=6.0):
    """Smooth TPR + FPR disparity across groups (scalar). Use rho = 6.

    Soft positive of individual i:  sigma(rho * s_i),  s_i = x_i^T w,
    sigma = logistic sigmoid (jax.nn.sigmoid). Per group s:
        TPR_s = sum_{i in G_s} sigma(rho s_i) * 1[y_i = 1] / sum_{i in G_s} 1[y_i = 1],
        FPR_s = sum_{i in G_s} sigma(rho s_i) * 1[y_i = 0] / sum_{i in G_s} 1[y_i = 0].
    With the given soft_max / soft_min:
        EOdds(w) = (1 / 2) * [ (soft_max(TPR) - soft_min(TPR))
                             + (soft_max(FPR) - soft_min(FPR)) ].
    """
    # Steps:
    #   1. soft_pos_s = jax.nn.sigmoid(rho * (X_list[s] @ w))  for every group s.
    #   2. stack the length-S vectors TPR_s and FPR_s using the formulas above.
    #   3. return the average of (soft_max - soft_min) over TPR and over FPR.
    raise NotImplementedError


def equal_opportunity(w, X_list, y_list, rho=6.0):
    """Smooth TPR disparity across groups (scalar) -- the TPR-only part of EOdds.

    With TPR_s exactly as in equalized_odds:
        EOpp(w) = soft_max(TPR) - soft_min(TPR).
    Use rho = 6.
    """
    # Steps: build the length-S vector TPR_s as in equalized_odds, then return
    #        soft_max(TPR, rho) - soft_min(TPR, rho).
    raise NotImplementedError

### Checkpoint 1: Compare Against the Toolbox

Fit the ERM model

$$
w^\star\left(\frac13,\frac13,\frac13\right)
$$

and compare your metric values against `badr.metrics`.

Agreement should be close to numerical precision. If a check fails, inspect the normalization first: common mistakes are summing over ordered instead of unordered group pairs, dividing by the wrong group count, or using hard labels instead of soft predictions.

In [ ]:
METRICS = {
    "Individual Fairness": individual_fairness,
    "Equalized Odds":      equalized_odds,
    "Equal Opportunity":   equal_opportunity,
}

w_erm = lh.fit_weights(model, dset, lh.uniform_weights(dset))
lh.check_close("Individual Fairness", individual_fairness(w_erm, X_list, y_list),
               badr.metrics.IndividualFairness().fun(w_erm, dset))
lh.check_close("Equalized Odds", equalized_odds(w_erm, X_list, y_list),
               badr.metrics.EqualizedOdds(rho=6.0).fun(w_erm, dset))
lh.check_close("Equal Opportunity", equal_opportunity(w_erm, X_list, y_list),
               badr.metrics.EqualOpportunity(rho=6.0).fun(w_erm, dset))

## 3. Question 2: Reweighting <a class="anchor" id="sec-reweight"></a>

Before optimizing $\lambda$, try a few fixed group weightings. This makes the bilevel problem concrete:

1. choose $\lambda$;
2. train $w^\star(\lambda)$;
3. evaluate fairness metrics and per-group losses;
4. compare the tradeoffs.

Available helper weightings:

- `lh.uniform_weights(dset)`: ERM-style weighting, $\lambda_s=1/S$;
- `lh.balanced_weights(dset)`: inverse group-size weighting, $\lambda_s\propto 1/n_s$;
- `jnp.eye(n_groups)[i]`: the simplex vertex that puts all weight on group $i$.

### Implementation

Complete `candidate_weightings(dset)`. Return a dictionary mapping readable names to valid simplex vectors. Include `"Uniform"`, `"Balanced"`, and one vertex for each group.

Before using the table, check that every vector is nonnegative and sums to one.

In [ ]:
def candidate_weightings(dset):
    """Return a dict {name: lambda} of group weightings to compare.

    Every lambda is a length-S vector on the simplex (entries >= 0, sum to 1):
        Uniform   : lambda_s = 1 / S                          -> lh.uniform_weights(dset)
        Balanced  : lambda_s = (1 / n_s) / sum_t (1 / n_t)     -> lh.balanced_weights(dset)
        Vertex e_i: lambda = e_i  (1 on group i, 0 elsewhere)  -> jnp.eye(S)[i]
    """
    # Build a dict {name: weight vector}; include "Uniform", "Balanced", and each
    # vertex e_i for i = 0 .. S - 1  (S = dset.n_groups).
    raise NotImplementedError

### Checkpoint 2: Inspect the Reweighting Table

`lh.print_weighting_table` fits $w^\star(\lambda)$ for each candidate weighting and reports all fairness metrics plus the worst-group loss

$$
\max_s \ell_s(w).
$$

Look for three things:

- whether the best weighting for one fairness metric is also best for the others;
- whether balanced weighting improves fairness or mainly changes the worst-group loss;
- whether putting all mass on one group helps that group at the expense of others.

This table should make clear why a fixed heuristic weighting is not enough for every fairness metric.

In [ ]:
weightings = candidate_weightings(dset)
lh.print_weighting_table(model, dset, METRICS, weightings)

## 4. Question 3: Min-Max Reweighting <a class="anchor" id="sec-minmax"></a>

Now compare the heuristic weights with a standard robust baseline: minimize the largest group loss,

$$
\min_w \max_s \ell_s(w).
$$

This objective is not the same as individual fairness, equalized odds, or equal opportunity. It is still useful because it directly controls the worst training loss across groups.

### Epigraph Form

Introduce a scalar $t$ and solve

$$
\min_{w,t}\ t
\quad\text{s.t.}\quad
\ell_s(w) + \frac{\mu}{2}\lVert w\rVert^2 \le t
\quad\text{for all }s.
$$

This is a convex problem for logistic regression and can be solved with CVXPY.

### Dual Weights

Each group constraint has a nonnegative dual variable $\alpha_s$. The normalized dual vector

$$
\lambda_s=\frac{\alpha_s}{\sum_t \alpha_t}
$$

acts as the group weighting associated with the min-max solution.

### Implementation

Complete `minmax_logreg(X_list, y_list, l2_reg)`. Return:

- `coef`: the fitted coefficient vector;
- `group_weights`: the normalized per-group dual variables.

Keep a handle to each group constraint so you can read its `dual_value` after solving. Clip tiny negative dual values to zero before normalizing, since conic solvers may return small numerical artifacts.

In [ ]:
import cvxpy as cp


def minmax_logreg(X_list, y_list, l2_reg=1e-1):
    """Min-max (worst-group) logistic regression via CVXPY.

    Per-group mean logistic loss (logistic(z) = log(1 + e^z) = cp.logistic):
        loss_s(w) = (1 / n_s) * sum_{i in G_s} [ logistic(x_i^T w) - y_i (x_i^T w) ].
    Convex epigraph program with shared L2 term (mu = l2_reg):
        minimize_{w, t}   t
        subject to        loss_s(w) + (mu / 2) ||w||_2 ** 2 <= t   for every group s.
    The normalized non-negative duals alpha_s >= 0 of the per-group constraints
    give the group weights:  group_weights_s = alpha_s / sum_t alpha_t.

    Returns (coef (d,), group_weights (S,)).
    """
    # Steps:
    #   1. w = cp.Variable(d), t = cp.Variable(), norm_var = cp.Variable(nonneg=True)
    #      with cp.SOC(norm_var, w) so norm_var = ||w||_2 and the term
    #      0.5 * l2_reg * norm_var ** 2  (i.e. (mu / 2) ||w||^2) stays convex.
    #   2. for each group: loss_s = cp.sum(cp.logistic(X @ w)
    #                                      - cp.multiply(y, X @ w)) / n_s;
    #      add the constraint  loss_s + 0.5 * l2_reg * norm_var ** 2 <= t  (keep its handle).
    #   3. solve cp.Problem(cp.Minimize(t), constraints) with solver="SCS".
    #   4. group_weights = per-group constraint duals (c.dual_value), clipped to
    #      >= 0, then normalized to sum to 1.
    raise NotImplementedError

### Checkpoint 3: Compare the Dual Weights

Run your CVXPY implementation and compare the recovered weights with `lh.minmax_reference`.

The tolerance is looser than in Question 1 because this problem is solved by a first-order conic solver. Check that the weights are nonnegative, sum to one, and put positive mass on the same groups as the reference.

In [ ]:
coef_mm, lam_minmax = minmax_logreg(dset.X_train_list, dset.y_train_list, l2_reg=1e-1)
_, lam_minmax_ref = lh.minmax_reference(dset, l2_reg=1e-1)
lh.check_close("min-max group weights", lam_minmax, lam_minmax_ref, atol=2e-2)
print("min-max lambda:", np.round(lam_minmax, 3))

## 5. Question 4: BADR Oracle and BADR-GD <a class="anchor" id="sec-oracle"></a>

BADR-GD updates three quantities:

- $w$, the current model parameters;
- $v$, an auxiliary vector approximating an inverse-Hessian product;
- $\lambda$, the group weights in the simplex.

One update has the form

$$
\begin{aligned}
w &\leftarrow w-\alpha\,\nabla_w f(w,\lambda),\\
v &\leftarrow v-\alpha\left(\nabla_w\mathrm{metric}(w)+\nabla^2_{ww}f(w,\lambda)\,v\right),\\
\lambda &\leftarrow \mathrm{Proj}_\Delta\left(\lambda-\beta\,J^\top v\right),
\end{aligned}
$$

where

$$
J=\partial_\lambda\nabla_w f(w,\lambda).
$$

The BADR implementation handles the iteration and simplex projection. You provide the derivative oracles:

| function | quantity | shape |
|---|---|---|
| `grad_lower_groups(parts, w, X_list, y_list)` | $\nabla_w\ell_s(w)$ for all groups $s$ | $(S,d)$ |
| `grad_upper(parts, w, lam)` | $(\nabla_w\mathrm{metric}(w),\ \partial_\lambda\mathrm{metric})$ | $(d,), (S,)$ |
| `hvp(parts, w, lam, v, X_list, y_list)` | $\left(\sum_s\lambda_s\nabla^2_w\ell_s(w)\right)v$ | $(d,)$ |
| `jt_v(parts, w, v, X_list, y_list)` | $\left[\langle\nabla_w\ell_s(w),v\rangle\right]_s$ | $(S,)$ |

The object `parts` stores compiled JAX derivative functions:

- `parts.grad_loss(w, X, y)` computes $\nabla_w\ell(w;X,y)$;
- `parts.hess_loss(w, X, y)` computes $\nabla_w^2\ell(w;X,y)$;
- `parts.grad_metric(w)` computes $\nabla_w\mathrm{metric}(w)$.

The metric has no direct dependence on $\lambda$, so $\partial_\lambda\mathrm{metric}(w)=0$.

### Implementation

Complete the four oracle functions below. Check shapes as you go. Most errors in this section come from returning `(d, S)` instead of `(S, d)`, returning Python lists instead of stacked arrays, or forgetting that `grad_upper` must also return a zero vector with the same shape as `lam`.

In [ ]:
def grad_lower_groups(parts, w, X_list, y_list):
    """Per-group lower-level gradients, stacked -> shape (S, d):

        return [ grad_w loss_s(w) ]_{s = 0 .. S-1}.

    Use parts.grad_loss(w, X_s, y_s) = grad_w loss(w; X_s, y_s) per group s.
    """
    # TODO
    raise NotImplementedError


def grad_upper(parts, w, lam):
    """Upper-level (metric) gradients -> (shape (d,), shape (S,)):

        return ( grad_w metric(w),  d metric / d lam ).

    The metric has no direct lam dependence, so d metric / d lam = 0 (shape (S,)).
    Use parts.grad_metric(w) = grad_w metric(w).
    """
    # TODO
    raise NotImplementedError


def hvp(parts, w, lam, v, X_list, y_list):
    """Hessian-vector product of the weighted lower-level objective -> shape (d,):

        return ( sum_s lam_s * hess_w loss_s(w) ) @ v.

    Use parts.hess_loss(w, X_s, y_s) = hess_w loss(w; X_s, y_s) per group s.
    """
    # TODO
    raise NotImplementedError


def jt_v(parts, w, v, X_list, y_list):
    """J^T v  with  J = d/dlam (grad_w f)  -> shape (S,):

        return [ <grad_w loss_s(w), v> ]_{s = 0 .. S-1}.

    Use parts.grad_loss(w, X_s, y_s) per group s.
    """
    # TODO
    raise NotImplementedError

### Checkpoint 4: Run BADR-GD for One Metric

Bundle the four oracle functions in `ORACLE_FNS` and run BADR-GD on **individual fairness**, warm-started from the ERM solution.

After the run, inspect `lam_badr`.

Questions to answer:

- Is the learned vector close to uniform, close to balanced, or concentrated near one group?
- Which part of the individual-fairness metric explains this movement?
- Would you expect the same $\lambda$ to be best for equalized odds?

In [ ]:
ORACLE_FNS = {
    "grad_lower_groups": grad_lower_groups,
    "grad_upper":        grad_upper,
    "hvp":               hvp,
    "jt_v":              jt_v,
}

lam_badr = lh.run_badr(dset, model, individual_fairness, ORACLE_FNS, w_erm,
                       name="Individual Fairness", verbose=True)
print("BADR lambda (Individual Fairness):", np.round(lam_badr, 3))

## 6. Final Experiment and Interpretation <a class="anchor" id="sec-figure"></a>

The final experiment repeats BADR-GD for each fairness metric and plots the result on the three-group simplex. Each panel represents a different outer objective

$$
V(\lambda)=\mathrm{metric}(w^\star(\lambda)).
$$

The triangle is colored by the metric value after retraining the model at that point. Colors are normalized separately in each panel. Lighter regions correspond to lower metric values.

Markers:

- `Uniform`: equal group weights;
- `Balanced`: inverse group-size weights;
- `Min-max`: the dual weighting from Question 3;
- `BADR`: the metric-specific weighting learned by BADR-GD.

Run the experiment and use the plots as an empirical map of the bilevel objective.

In [ ]:
panels = []
for name, metric_fn in METRICS.items():
    print(f"[{name}]")
    lam = lh.run_badr(dset, model, metric_fn, ORACLE_FNS, w_erm, name=name)
    markers = lh.default_markers({
        "Uniform":  lh.uniform_weights(dset),
        "Balanced": lh.balanced_weights(dset),
        "Min-max":  lam_minmax,
        "BADR":     lam,
    })
    panels.append((name, metric_fn, markers))

fig = lh.plot_simplex_figure(
    dset, model, panels, grid_n=45,
    title="BADR on the group-weight simplex  ·  ACSIncome, Alaska 2018",
)

### Questions to Answer <a class="anchor" id="sec-discussion"></a>

Answer these questions from the final table and simplex plots.

1. For each fairness metric, does BADR move toward a lower-value region of the simplex?
2. Does the min-max weighting help all fairness metrics equally, or is it aligned with only some of them?
3. Compare Uniform and Balanced. Which one is more competitive on this dataset, and what does that suggest about group-size imbalance?
4. Are the BADR optima for individual fairness, equalized odds, and equal opportunity in the same part of the simplex? Explain.
5. If you repeated the lab on another dataset, which parts of the workflow would stay the same and which conclusions might change?

### Extensions

Natural extensions are:

- replace the metric with `badr.metrics.DemographicParity`, `GroupVariance`, `HSIC`, or `DisparateMistreatment`;
- repeat the study on `fetch_adult`, `fetch_compas`, or `fetch_lawschool`;
- compare the stochastic oracle used here with the implicit oracle;
- use the high-level `badr.Badr` interface once the derivative structure is clear.

<div align="right"><a href="#toc">Back to TOC</a></div>